<a href="https://colab.research.google.com/github/kimimilo/hello-world-/blob/main/DAA_Topic3_SQLLab2_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import pandas as pd

# An in-memory SQLite database: it lives only while this notebook runtime is running
conn = sqlite3.connect(":memory:")

def sql(query):
    """Run a SELECT query and show the result as a table."""
    return pd.read_sql_query(query, conn)

def run(statement):
    """Run statements that change the database (CREATE, INSERT, UPDATE, DELETE)."""
    conn.executescript(statement)
    conn.commit()

print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.45.1


In [2]:
# when creating tables indents are not necessary but makes it easier to read
# for SQLite main data types : INTEGER, FLOAT, REAL, BLOB, NULL, TEXT

# ) ends e table definition
# ; ends the SQL command
# when inserting values must follow the column order

# drop table shi v important as we alrdy ran the table once
run("""
DROP TABLE IF EXISTS Customers;
DROP TABLE IF EXISTS Orders;

CREATE TABLE Customers (
    CustomerID INTEGER PRIMARY KEY,
    Name       TEXT,
    Segment    TEXT
);

CREATE TABLE Orders (
    OrderID    INTEGER PRIMARY KEY,
    CustomerID INTEGER,
    Amount     REAL,
    OrderDate  TEXT
);

INSERT INTO Customers VALUES
    (1, 'Alice', 'Premium'),
    (2, 'Bob',   'Standard'),
    (3, 'Chloe', 'Premium'),
    (4, 'Dinesh','Standard'),
    (5, 'Evan',  'Standard');

INSERT INTO Orders VALUES
    (101, 1, 250.0, '2026-01-05'),
    (102, 1, 120.0, '2026-02-11'),
    (103, 2,  80.0, '2026-02-14'),
    (104, 3, 560.0, '2026-03-02'),
    (105, 4,  45.5, '2026-03-09'),
    (106, 9,  99.9, '2026-03-15'),
    (107, 3, 210.0, '2026-04-01');
""")

display(sql("SELECT * FROM Customers"))
sql("SELECT * FROM Orders")

# * means to select all e columns from a specific table

,CustomerID,Name,Segment
0,1,Alice,Premium
1,2,Bob,Standard
2,3,Chloe,Premium
3,4,Dinesh,Standard
4,5,Evan,Standard


,OrderID,CustomerID,Amount,OrderDate
0,101,1,250.0,2026-01-05
1,102,1,120.0,2026-02-11
2,103,2,80.0,2026-02-14
3,104,3,560.0,2026-03-02
4,105,4,45.5,2026-03-09
5,106,9,99.9,2026-03-15
6,107,3,210.0,2026-04-01


In [3]:
# INNER JOIN: returns only rows where a match exists in both tables
# no match? row dropped

# SQL processes the FROM & JOIN before select thats why it's defined after using the aliases
# define aliases same line as the FROM/JOIN operations
# INNER JOIN combines C & O & matches rows where the customer ID in both tables are e same
# ON used w JOIN to tell SQL how tables should be connected

# inner join on here matches a C w an O when their CustomerID values are equal

sql("""
SELECT c.Name, c.Segment, o.OrderID, o.Amount
FROM Customers c
INNER JOIN Orders o ON c.CustomerID = o.CustomerID
ORDER BY c.Name
""")

,Name,Segment,OrderID,Amount
0,Alice,Premium,101,250.0
1,Alice,Premium,102,120.0
2,Bob,Standard,103,80.0
3,Chloe,Premium,104,560.0
4,Chloe,Premium,107,210.0
5,Dinesh,Standard,105,45.5


In [4]:
# LEFT JOIN: every row from the left table + matches from the right
# unmatched columns results in NULL

# left table is the table written before LEFT JOIN
# SQL auto uses ASC by default btw

sql("""
SELECT c.Name, o.OrderID, o.Amount
FROM Customers c
LEFT JOIN Orders o ON c.CustomerID = o.CustomerID
ORDER BY c.CustomerID
""")

# Nan is panda's way of saying NULL

,Name,OrderID,Amount
0,Alice,102.0,120.0
1,Alice,101.0,250.0
2,Bob,103.0,80.0
3,Chloe,107.0,210.0
4,Chloe,104.0,560.0
5,Dinesh,105.0,45.5
6,Evan,NaN,NaN


In [5]:
# RIGHT JOIN: every row from the right table + matches from the left
# right table is the table written
# RIGHT JOIN keeps every order, including the orphan order 106

sql("""
SELECT c.Name, o.OrderID, o.Amount
FROM Customers c
RIGHT JOIN Orders o ON c.CustomerID = o.CustomerID
ORDER BY o.OrderID
""")


,Name,OrderID,Amount
0,Alice,101,250.0
1,Alice,102,120.0
2,Bob,103,80.0
3,Chloe,104,560.0
4,Dinesh,105,45.5
5,None,106,99.9
6,Chloe,107,210.0


In [6]:
# Same result written as a LEFT JOIN: swap the table order

sql("""
SELECT c.Name, o.OrderID, o.Amount
FROM Orders o
LEFT JOIN Customers c ON c.CustomerID = o.CustomerID
ORDER BY o.OrderID
""")

# the orphan order 106 has no customer name

,Name,OrderID,Amount
0,Alice,101,250.0
1,Alice,102,120.0
2,Bob,103,80.0
3,Chloe,104,560.0
4,Dinesh,105,45.5
5,None,106,99.9
6,Chloe,107,210.0


In [7]:
# Query design: how SQL evaluates a query
# can't use an aggregate in WHERE as groups don't exist yet
# can't use a SELECT alias inside WHERE as SELECT is processed b4 WHERE

# FROM & JOIN: build the base dataset from the joined tables
# WHERE: filter individual rows
# GROUP BY: collapse rows into groups
# HAVING: filter groups
# SELECT: choose/compute the output columns
# ORDER BY then LIMIT: sort and

# where c.segment keeps only premium customers
# GROUP BY compiles all orders belonging to the same Name, allowing SUM() to calculate one total/cusotmer
sql("""
SELECT c.Name, SUM(o.Amount) AS total_spent
FROM Customers c
JOIN Orders o ON c.CustomerID = o.CustomerID
WHERE c.Segment = 'Premium'
GROUP BY c.Name
ORDER BY total_spent DESC
""")

,Name,total_spent
0,Chloe,770.0
1,Alice,370.0


In [8]:
# Lab task: customer lifetime value
# 2 new functions: COALESCE(x, 0) replaces NULL with 0 & COUNT(o.OrderID) counts only non-NULL order IDs, so Evan correctly gets 0

clv = sql("""
SELECT c.Name,
       c.Segment,
       COUNT(o.OrderID)                 AS orders,
       COALESCE(SUM(o.Amount), 0)       AS lifetime_value,
       ROUND(AVG(o.Amount), 2)          AS avg_order_value,
       MIN(o.OrderDate)                 AS first_order,
       MAX(o.OrderDate)                 AS last_order
FROM Customers c
LEFT JOIN Orders o ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.Name, c.Segment
ORDER BY lifetime_value DESC
""")
clv

# Because sql() returns a pandas DataFrame, you can continue in Python, e.g. clv.groupby("Segment")["lifetime_value"].mean(). SQL and pandas work together in real pipelines

,Name,Segment,orders,lifetime_value,avg_order_value,first_order,last_order
0,Chloe,Premium,2,770.0,385.0,2026-03-02,2026-04-01
1,Alice,Premium,2,370.0,185.0,2026-01-05,2026-02-11
2,Bob,Standard,1,80.0,80.0,2026-02-14,2026-02-14
3,Dinesh,Standard,1,45.5,45.5,2026-03-09,2026-03-09
4,Evan,Standard,0,0.0,NaN,None,None


In [9]:
# CRUD: creating w INSERT
# INSERT rules: values must match e column order and types
  # listing the columns explicitly, INSERT INTO Customers (CUstomerID, Name, Segment)

run("INSERT INTO Customers (CustomerID, Name, Segment) VALUES (6, 'Fiona', 'Standard')")
sql("SELECT * FROM Customers WHERE CustomerID = 6")

,CustomerID,Name,Segment
0,6,Fiona,Standard


In [12]:
# Inserting a duplicate primary key fails, which protects your data
# We can use INSERT OR IGNORE to prevent the IntegrityError if the ID already exists
run("INSERT OR IGNORE INTO Customers VALUES (6, 'Another Fiona', 'Premium')")
# = an error because of UNIQUE constraint such as duplicate PK

# Or we can verify who is currently customer 6
sql("SELECT * FROM Customers WHERE CustomerID = 6")

,CustomerID,Name,Segment
0,6,Fiona,Standard


In [13]:
# CRUD: UPDATE & DELETE safely
# alw include a WHERE clause, w/o it UPDATE overwrites every row & DELETE empties e whole table
    # which is insant & irreversible btw

# write e WHERE clause as a SELECT first then check exactly which rows it returns

# 1. Preview: which row will change?
sql("SELECT * FROM Customers WHERE CustomerID = 2")

,CustomerID,Name,Segment
0,2,Bob,Standard


In [14]:
 # only then swap SELECT * for UPDATE ... SET or DELETE, keeping the same WHERE
 # use sql() when iw to retrieve/display data using SELECT
 # use run() when iw to chnage e database using INSERT, UPDATE, DELETE

 # 2. Update with the same WHERE, 3. verify
run("UPDATE Customers SET Segment = 'VIP' WHERE CustomerID = 2")

# general SQL format:
    # UPDATE table_name SET column_name = new_value WHERE condition;
    # new_value means e replacement value that can be any data type


sql("SELECT * FROM Customers WHERE CustomerID = 2")
# run a SELECT afterwards to verify

,CustomerID,Name,Segment
0,2,Bob,VIP


In [15]:
# DELETE safely
# Clean up the orphan order found earlier
sql("SELECT * FROM Orders WHERE OrderID = 106")

,OrderID,CustomerID,Amount,OrderDate
0,106,9,99.9,2026-03-15


In [16]:
# code DELETES order 106 & then counts how many orders remain
run("DELETE FROM Orders WHERE OrderID = 106")
sql("SELECT COUNT(*) AS remaining_orders FROM Orders")

,remaining_orders
0,6


In [17]:
# Customers with total spend above 300 (hint: HAVING)

sql("""
SELECT c.Name, SUM(o.Amount) AS total_spent
FROM Customers c JOIN Orders o ON c.CustomerID = o.CustomerID
GROUP BY c.Name
HAVING SUM(o.Amount) > 300
""")

# HAVING used to filter groups after SQL calculates their totals
# used here as the conidtion is based on the aggregate function SUM()

,Name,total_spent
0,Alice,370.0
1,Chloe,770.0


In [18]:
# Monthly revenue (hint: substr(OrderDate, 1, 7) extracts YYYY-MM)
# substr() means "substring", which means taking only part of a piece of text

sql("""
SELECT substr(OrderDate, 1, 7) AS month, COUNT(*) AS orders, SUM(Amount) AS revenue
FROM Orders
GROUP BY month
ORDER BY month
""")

,month,orders,revenue
0,2026-01,1,250.0
1,2026-02,2,200.0
2,2026-03,2,605.5
3,2026-04,1,210.0
